<a href="https://colab.research.google.com/github/Sruthi-Gudelli/Comment-Toxicity-Detection/blob/main/BERT_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_data = pd.read_csv('/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/train.csv')
train_data.head()

In [ ]:
X = train_data['comment_text']
y = train_data[['toxic', 'severe_toxic', 	'obscene', 	'threat', 'insult',	'identity_hate']]

**Splitting the data**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.values
X_test = X_test.values
y_train = y_train.reset_index(drop=True).values
y_test = y_test.reset_index(drop=True).values

**Training BERT Model**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

In [ ]:
# Creating the Dataset Assembly Line
class ToxicityDatasetBERT(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])

        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[index], dtype=torch.float32)
        }

In [ ]:
# Building the Many-to-One BERT Model
class BERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_classes=6):
        super(BERTMultiLabelClassifier, self).__init__()
        # Loads pre-trained BERT weights (110 million parameters)
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes) # hidden_size = 768

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Isolates the [CLS] token summary vector
        pooled_output = outputs.pooler_output

        out = self.dropout(pooled_output)
        return self.fc(out) # Returns 6 raw logits (No Sigmoid here!)

In [ ]:
# Tokenizing data & Creating Loaders
bert_tokenizer_tool = BertTokenizer.from_pretrained('bert-base-uncased')

# Creating PyTorch datasets and loaders
train_dataset = ToxicityDatasetBERT(X_train, y_train, bert_tokenizer_tool, max_len=128)
test_dataset = ToxicityDatasetBERT(X_test, y_test, bert_tokenizer_tool, max_len=128)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

**Training Loop with Mixed Precision(fp16)**
Because BERT is massive, using PyTorch's Automatic Mixed Precision (torch.cuda.amp) reduces numerical calculations from 32-bit to 16-bit, cutting VRAM usage in half and doubling training speeds with zero loss in accuracy.

In [ ]:
import torch.amp as amp
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERTMultiLabelClassifier().to(device)

# BERT requires a very small learning rate (2e-5) or it breaks
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

# Scaler for Automatic Mixed Precision (amp)
scaler = torch.amp.GradScaler() # Fixes vanishing gradient issues on 16-bit math [1]

EPOCHS = 4 # BERT converges incredibly fast, usually 1 or 2 epochs is enough

for epoch in range(EPOCHS):

    # TRAINING PHASE
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        # 16-bit math for fast training
        with amp.autocast(device_type='cuda'):
            predictions = model(input_ids, attention_mask)
            loss = criterion(predictions, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        # Training accuracy
        probs = torch.sigmoid(predictions)
        preds_binary = (probs >= 0.5).float()
        train_correct += (preds_binary == labels).sum().item()
        train_total += labels.numel()

    # Testing PHASE

    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0

    # torch.no_grad() stops PyTorch from saving math memory, saving massive VRAM
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with amp.autocast(device_type='cuda'):
                test_preds = model(input_ids, attention_mask)
                t_loss = criterion(test_preds, labels)

            test_loss += t_loss.item()

            # Test accuracy
            t_probs = torch.sigmoid(test_preds)
            t_preds_binary = (t_probs >= 0.5).float()
            test_correct += (t_preds_binary == labels).sum().item()
            test_total += labels.numel()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = test_loss / len(test_loader)
    train_accuracy = (train_correct / train_total) * 100
    test_accuracy = (test_correct / test_total) * 100

    print(f"Epoch {epoch+1:04} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}% | Val Loss: {avg_val_loss:.4f} | Test Acc: {test_accuracy:.2f}%")


**Saving the model**

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/bert_model.pt")
bert_tokenizer_tool.save_pretrained("/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/bert_tokenizer")
print("Weights and tokenizer saved successfully!")


**Predicting with new test data**

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import BertTokenizer, BertModel
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler

class BERTMultiLabelClassifier(nn.Module):
    def __init__(self, num_classes=6):
        super(BERTMultiLabelClassifier, self).__init__()
        # Loads pre-trained BERT weights (110 million parameters)
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes) # hidden_size = 768

    def forward(self, input_ids, attention_mask):
        device = input_ids.device
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Isolates the [CLS] token summary vector
        pooled_output = outputs.pooler_output

        out = self.dropout(pooled_output)
        return self.fc(out)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTMultiLabelClassifier()
model.load_state_dict(torch.load("/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/bert_model.pt", map_location=device))
model.to(device)
model.eval()


tokenizer = BertTokenizer.from_pretrained("/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/bert_tokenizer")

file_path = "/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/test.csv"
df = pd.read_csv(file_path)
new_sentences = df['comment_text'].tolist()

# Tokenizing raw text data
encoded_data = tokenizer(
    new_sentences,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
batch_size = 32  # Low batch size prevents RAM crashes
dataset = TensorDataset(encoded_data['input_ids'], encoded_data['attention_mask'])
sampler = SequentialSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler, batch_size=batch_size)

all_predictions = []

with torch.no_grad():
    for batch in dataloader:
        b_input_ids = batch[0].to(device)
        b_attn_mask = batch[1].to(device)

        outputs = model(input_ids=b_input_ids, attention_mask=b_attn_mask)
        logits = outputs if not isinstance(outputs, tuple) else outputs

        # Calculate probabilities and binary choices for this small batch
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities > 0.5).int()

        # Collect batch results
        all_predictions.append(predictions)

final_preds_matrix = torch.cat(all_predictions, dim=0).cpu().numpy()
column_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
pred_df = pd.DataFrame(final_preds_matrix, columns=column_names)


In [ ]:
pred_df.insert(0, 'id', df['id'].values)


In [ ]:
final_Btest_data = pd.merge(df, pred_df, on = 'id', how = 'inner')
final_Btest_data.head()

In [ ]:
final_Btest_data.to_csv('/content/drive/MyDrive/GUVI-DS-PRACTICE/datasets/Final-BERT-TestData.csv')